# VMC2026 Track 3 — Baseline (ECAPA speaker/accent similarity)

Cho **cặp** (wav_a, wav_b) → dự đoán `spk_sim` (giống người nói) + `acc_sim` (giống accent).
Code + **checkpoint pre-trained** có sẵn trong repo (Baseline 2 fine-tuned).

Điểm dev tham khảo: spk SRCC ~0.45, acc SRCC ~0.44. Format nộp `answer.txt`.

> ⚠️ **CẦN GỘP WAV CỦA VCTK.** Gói `_syn` KHÔNG có **sys004, sys008, sys019, sys021**
> (`wav_b` reference luôn là sys019). Tải gói VCTK riêng (README track3), upload thành dataset
> thứ 2, rồi copy `wav/` của nó vào chung `wav/` trước khi inference — không sẽ thiếu file.

## 0. Config + GỘP data (tự dò _syn & _vctk → gộp wav vào /kaggle/working/track3-data)
Add Input: dataset chứa gói `_syn` **và** gói `_vctk` (Kaggle tự giải nén .tar.gz). Cell tự copy + gộp → đủ **3.548 wav**.

In [ ]:
import os, glob, shutil, tarfile

def find_one(*patterns):
    for p in patterns:
        hits = glob.glob(p, recursive=True)
        if hits: return sorted(hits)[0]
    return None

def ensure_pkg(name):
    """Trả về thư mục gói `name`. Kaggle đã giải nén → dùng luôn; chỉ có .tar.gz → tự giải nén ra working."""
    d = find_one(f'/kaggle/input/**/{name}')
    if d:
        return d
    tar = find_one(f'/kaggle/input/**/{name}.tar.gz')
    assert tar, f'Không thấy {name} (cả thư mục lẫn .tar.gz) — đã Add Input dataset chưa?'
    out = f'/kaggle/working/_ex/{name}'
    if not os.path.exists(out):
        os.makedirs(out, exist_ok=True)
        print('Giải nén', os.path.basename(tar), '...')
        with tarfile.open(tar) as tf:
            tf.extractall(out)
    return find_one(f'{out}/{name}', f'{out}/**/{name}') or out

SYN_SRC  = ensure_pkg('vmc2026_track3_train_phase_distro_v3_syn')
VCTK_SRC = ensure_pkg('vmc2026_track3_train_phase_distro_v3_vctk')

# Chẩn đoán cấu trúc (in ra để biết wav nằm đâu)
print('SYN :', SYN_SRC, '| nội dung:', sorted(os.listdir(SYN_SRC))[:8])
print('VCTK:', VCTK_SRC, '| nội dung:', sorted(os.listdir(VCTK_SRC))[:8])

# Dò đúng thư mục wav của SYN (phòng trường hợp lồng thêm 1 cấp)
if not os.path.isdir(f'{SYN_SRC}/wav'):
    cand = find_one(f'{SYN_SRC}/**/wav')
    assert cand, f'Không thấy thư mục wav trong {SYN_SRC} — kiểm tra lại cấu trúc gói _syn.'
    SYN_SRC = os.path.dirname(cand)
    print('→ Điều chỉnh SYN_SRC =', SYN_SRC)
print('SYN/wav số file:', len(glob.glob(f'{SYN_SRC}/wav/*.wav')))

vctk_wavs = glob.glob(f'{VCTK_SRC}/wav/*.wav') or glob.glob(f'{VCTK_SRC}/**/*.wav', recursive=True)
print('VCTK wav số file:', len(vctk_wavs))

# ── Dựng thư mục làm việc SẠCH (xoá tàn dư run trước → tránh lỗi thiếu wav/) ──
DATA_ROOT = '/kaggle/working/track3-data'
shutil.rmtree(DATA_ROOT, ignore_errors=True)
print('Copy gói _syn ra working...')
shutil.copytree(SYN_SRC, DATA_ROOT)
os.makedirs(f'{DATA_ROOT}/wav', exist_ok=True)   # chắc chắn có wav/ trước khi gộp VCTK

# ── GỘP wav VCTK vào wav/ chung ──
n_vctk = 0
for w in vctk_wavs:
    dst = f'{DATA_ROOT}/wav/{os.path.basename(w)}'
    if not os.path.exists(dst):
        shutil.copy(w, dst); n_vctk += 1
total = len(glob.glob(f'{DATA_ROOT}/wav/*.wav'))
print(f'Đã gộp VCTK: {n_vctk} file · TỔNG wav: {total} (kỳ vọng 3548)')

# ── Đường dẫn cho inference ──
DEV_CSV  = f'{DATA_ROOT}/sets/dev.csv'
T3       = '/kaggle/working/vmc2026-baselines/track3'
OUT_DIR  = '/kaggle/working'
CKPT_SPK = f'{T3}/official-egs/spk_sim_adamw_lr1e-3/model_spk_sim_step20000.pt'
CKPT_ACC = f'{T3}/official-egs/acc_sim_adamw_lr1e-3/model_acc_sim_step20000.pt'
print('DEV_CSV:', DEV_CSV, '· tồn tại:', os.path.exists(DEV_CSV))

## 1. Cài đặt
Repo gốc dùng `uv`; trên Kaggle cài speechbrain + chạy python trực tiếp. Nếu thiếu dep, xem `track3/pyproject.toml`.

In [ ]:
!git clone -q https://github.com/voicemos-challenge/vmc2026-baselines.git /kaggle/working/vmc2026-baselines
# CHỈ cài speechbrain — KHÔNG cài lại torch/torchaudio (Kaggle đã có sẵn, cài thêm dễ vỡ CUDA).
!pip install -q speechbrain

# Kiểm tra torch/torchaudio dùng được + checkpoint đã tải đủ (file thật ~86MB, không phải LFS pointer)
import torch, torchaudio
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__, '| CUDA', torch.cuda.is_available())
for ck in [CKPT_SPK, CKPT_ACC]:
    sz = os.path.getsize(ck) / 1e6 if os.path.exists(ck) else 0
    print(f'{"✅" if sz > 1 else "❌"} {os.path.basename(ck)}: {sz:.1f} MB')
    assert sz > 1, 'Checkpoint thiếu/là LFS pointer — clone lại hoặc kiểm tra repo.'

## 2. Inference — speaker + accent similarity (ECAPA fine-tuned)
Chạy `inference.py` 2 lần (spk + acc) với checkpoint Baseline 2. GPU T4 + Internet On (tải model SpeechBrain).

In [ ]:
# Mỗi metric chạy riêng. BẮT BUỘC truyền --target-metric đúng để load đúng head của checkpoint.
!cd {T3} && python inference.py --data-root {DATA_ROOT} --csv-path {DEV_CSV} --checkpoint {CKPT_SPK} --target-metric spk_sim --out {OUT_DIR}/spk_dev.csv
!cd {T3} && python inference.py --data-root {DATA_ROOT} --csv-path {DEV_CSV} --checkpoint {CKPT_ACC} --target-metric acc_sim --out {OUT_DIR}/acc_dev.csv

## 3. Gộp spk + acc → answer.txt
⚠️ Kiểm tra tên cột điểm thực trong output rồi chỉnh `SPK_COL`/`ACC_COL`.

In [ ]:
import pandas as pd
SPK_COL, ACC_COL = 'pred_spk_sim', 'pred_acc_sim'   # khớp output inference.py (pred_<metric>)
KEYS = ['system_id', 'utterance_id', 'wav_a_path', 'wav_b_path']

spk = pd.read_csv(f'{OUT_DIR}/spk_dev.csv').rename(columns={SPK_COL: 'pred_spk_sim'})
acc = pd.read_csv(f'{OUT_DIR}/acc_dev.csv').rename(columns={ACC_COL: 'pred_acc_sim'})
print('spk rows:', len(spk), '| acc rows:', len(acc), '(kỳ vọng 600 mỗi file)')

merged = spk.merge(acc[KEYS + ['pred_acc_sim']], on=KEYS, how='outer')
cols = KEYS + ['pred_acc_sim', 'pred_spk_sim']

# An toàn: nếu có cặp lỗi inference → thiếu điểm (NaN) → cảnh báo + điền 3.0 cho file hợp lệ
na = merged[['pred_acc_sim', 'pred_spk_sim']].isna().sum().sum()
if na:
    print(f'⚠️ {na} ô thiếu điểm (có pair lỗi lúc inference) → điền tạm 3.0. Nên xem lại log inference.')
    merged[['pred_acc_sim', 'pred_spk_sim']] = merged[['pred_acc_sim', 'pred_spk_sim']].fillna(3.0)

merged[cols].to_csv(f'{OUT_DIR}/answer.txt', index=False)
print(f'Ghi {len(merged)} dòng → answer.txt (kỳ vọng 600)')
merged[cols].head()

## 4. Đóng zip nộp

In [ ]:
!cd {OUT_DIR} && zip -j submission_track3.zip answer.txt && unzip -l submission_track3.zip